In [29]:
# Feature Extraction and Engineering
import re, json, time, pickle

from pandas import value_counts
from sklearn.model_selection import KFold
import os, sys, warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

CSV_PATH = os.environ.get("LOAN_CSV", "dataset/SingleCycleLoaneeMember.csv")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
OUT = os.path.join("./", "results"); os.makedirs(OUT, exist_ok=True)

OUT = os.path.join('./', "results"); os.makedirs(OUT, exist_ok=True)
MODELS = os.path.join('./', "models"); os.makedirs(MODELS, exist_ok=True)

# From Base Model
LADDER = np.array([5, 10, 20, 25, 30, 40, 50, 60, 70, 80, 90, 100, 120, 150,
                   200, 250, 300, 400, 500, 800, 1000, 2000, 5000]) * 1000

def rung(v):
    return np.abs(np.log(np.asarray(v, float))[:, None] - np.log(LADDER)[None, :]).argmin(1)


EDU_MAP = {"Illiterate": 0, "Literacy": 1, "Pre-Primary": 2, "Primary": 3, "PSC": 3, "JSC": 4,
           "Secondary": 4, "SSC": 5, "Higher-Secondary": 6, "HSC": 6, "Graduation": 7,
           "Honors": 7, "Degree": 7, "Post-Graduation": 8, "Masters": 8, "PHD": 9}
BIG = 300000


def load():
    df = pd.read_csv(CSV_PATH)
    df["_adm"] = pd.to_datetime(df["admission_date"])
    return df.sort_values(["_adm", "member_id"]).reset_index(drop=True)


def metrics(actual, pred):
    a = np.asarray(actual, float); p = np.clip(np.asarray(pred, float), 4000, 5e6)
    la, lp = np.log1p(a), np.log1p(p); big = a >= 300000
    return {"MAE": float(np.mean(np.abs(p - a))), "RMSLE": float(np.sqrt(np.mean((lp - la) ** 2))),
            "R2_log": float(1 - np.sum((la - lp) ** 2) / np.sum((la - la.mean()) ** 2)),
            "R2_taka": float(1 - np.sum((a - p) ** 2) / np.sum((a - a.mean()) ** 2)),
            "within_1_step": float(np.mean(np.abs(rung(p) - rung(a)) <= 1)),
            "exact_step": float(np.mean(rung(p) == rung(a))),
            "MAE_big(>=300k)": float(np.mean(np.abs(p - a)[big])), "MAE_normal": float(np.mean(np.abs(p - a)[~big])),
            "AUC_big": float(roc_auc_score(big, np.log(p)))}

- 386 member request low loan than but have better income
- First predict loan_component with a model (general and agrosor)
- Then two more model one is for general loan amount another for aggrosor ( separated model not work well that with component model)
- Expense bellow than 10% of Income, 12.3% member have that issue.
- Standard products: Loan amount ≤ ⅓ of yearly income.
- AGROSOR: Loan amount ≤ ⅔ of yearly income.


- Condition to check: general 100 member history and calculate this ratio
- Q50,the typical amount for similar members.
- Q90 a plausible ceiling
- P(big), the AGROSOR eligibility probability. ( after calculation)
-history of officers cutting amounts ( request amount to approve amount)

Ceiling=min(Q90, IncomeCap, ProductMax)   #### human work
stretch = requested amount / Q50(profile)
18.2% of actual AGROSOR requests have P(big) below 0.2, so their profile weakly supports the request , and 2.2% of non-big requests (505 loans) have P(big) above 0.5.

Income below 10% expenses was not more common in the red zone (10.3% vs 12.3% overall), so bad income entries don’t explain the stretches.

#### Divided loan request in 4 steps
1. Below P10
2. Green
3. Amber
4. Red

In [28]:
data = load()
a=data[data['loan_amount'] >= 300000].copy()
a['loan_Component'].value_counts()

loan_Component
AGROSOR          10405
AGROSOR-RAISE       92
AGROSOR-MFCE        75
Name: count, dtype: int64

In [6]:
data.head()

,member_id,name,dateofbirth,gender,current_addr_district,current_addr_thana,maritial_status,admission_date,education,occupation,alternative_occupation,total_family_member,total_earning_member,total_monthly_income,yearly_total_income,yearly_total_expenses,yearly_total_savings,home_owner_type,year_of_rent,farm_land_amt,loan_id,proposer_oldid,loan_cycle_no,loan_Component,loan_use_sources,applied_loan_amount,loan_amount,installmenttype,installmentyear,totalInstallmentcount
0,1339749,NAJMA BEGUM,1984-01-01,Female,Bhola,Bhola Sadar,Married,2026-06-03,Secondary,Housewife,NaN,4,2,100000.0,1200000.0,450000.0,750000.0,Own,0.0,50,2219692,ID Nai,1,JAGORON,Dairy Farm (ডেইরী ফার্ম/গাভী পালন),90000.0,90000.0,Mounthly,1.0,12
1,1336203,SABINA KHATUN,1996-08-04,Female,Faridpur,Modhukhali,Married,2026-05-17,Pre-Primary,Housewife,NaN,4,1,40000.0,480000.0,320000.0,160000.0,Own,0.0,90,2210543,01569126132,1,SUFOLON,Jute cultivation (পাট চাষ),80000.0,80000.0,Mounthly,1.0,12
2,1335120,RAHIMA AKTER,1988-11-10,Female,Shariatpur,Damuddya,Married,2026-05-14,Pre-Primary,Housewife,NaN,5,1,450000.0,5400000.0,400000.0,5000000.0,Own,0.0,10,2203667,01751090505,1,JAGORON,Dairy Farm (ডেইরী ফার্ম/গাভী পালন),200000.0,200000.0,Mounthly,1.0,12
3,872455,Mst Forida khatun,1982-03-25,Female,Meherpur,Meherpur,Married,2026-05-11,Primary,Housewife,NaN,4,2,80000.0,960000.0,500000.0,460000.0,Own,0.0,20,2205405,0187240723,1,JAGORON,Banana Cultivation (কলা চাষ),200000.0,200000.0,Weekly,1.0,46
4,1269453,Md Shahinur Islam,1982-03-19,Male,Dinajpur,Ghoraghat,Married,2026-01-01,Pre-Primary,Business,NaN,3,1,30000.0,360000.0,144000.0,216000.0,Own,0.0,200,2200795,55060708,1,JAGORON,Dairy Farm (ডেইরী ফার্ম/গাভী পালন),200000.0,200000.0,Mounthly,1.0,12


In [3]:
def clean_text_purpose(s: pd.Series) -> pd.Series:
    return s.fillna("NA").astype(str).str.replace(r"\s*\(.*$", "", regex=True).str.strip()

In [4]:
def purpose_group(s: str) -> str:
    s = s.lower()
    if re.search(r"immigrant|abroad", s): return "Migration"
    if re.search(r"dairy|goru|cattle fatt|cow|goat|poultry|duck|sheep|livestock|animal|motataja|rearing|pigeon|farm(ing)?$", s): return "Livestock"
    if re.search(r"fish|shrimp|pond", s): return "Fishery"
    if re.search(r"paddy|vegetable cult|potato|jute|onion|betel|corn|maize|cultiv|wheat|mustard|chili|garlic|banana|mango|tobacco|crop|nursery|agricultur|seed", s): return "Crop"
    if re.search(r"house|home|building|repair|toilet|latrine|tube|solar|sanitation|water", s): return "Housing"
    if re.search(r"auto|vehicle|rickshaw|van|transport|truck|cng|bus$|tempo|motor|bike|boat|driver|easy bike|battery", s): return "Transport"
    if re.search(r"business|shop|store|trade|dokan|stall|tea|grocery|cloth|garment|hardware|restaurant|hotel|medicine|pharmacy|distributor|mobile|electric|jewel|crockery|sanitary|brick|saw|mill|tailor|cosmetic|furniture|fertilizer|feed|wholesale|retail|bakery|food|fast|sweet|salon|parlor|barber|gas|plastic|glass", s): return "Trade"
    return "Other"

In [11]:
pur =clean_text_purpose (data['loan_use_sources'])
pur.map(purpose_group).value_counts()

loan_use_sources
Livestock    40018
Crop         35396
Trade        20035
Transport     9367
Fishery       7135
Migration     5440
Housing       4732
Other         1870
Name: count, dtype: int64

In [13]:
def altocc_group(s):
    if pd.isna(s): return "none"
    s = str(s).strip().lower()
    if re.search(r"agri|krish|krisi|farm|kisi|dairy|fish|cultiv", s): return "agri"
    if re.search(r"bus|shop|dokan|trad", s): return "business"
    if re.search(r"job|serv", s): return "job"
    if re.search(r"probas|abroad|foreign|immig", s): return "abroad"
    if re.search(r"house|home", s): return "housewife"
    if re.search(r"driv|auto|rickshaw|van", s): return "driver"
    return "other"

In [15]:
data["alternative_occupation"].map(altocc_group).value_counts()

alternative_occupation
none         80745
agri         20812
business      7543
other         7480
job           2892
housewife     1840
abroad        1581
driver        1100
Name: count, dtype: int64

In [17]:
def clean_id(s: pd.Series) -> pd.Series:
    return s.astype("string").str.strip().str.lstrip("0").fillna("NA").astype(str)

In [19]:
clean_id(data["proposer_oldid"]).value_counts()

proposer_oldid
NA           3995
74720110      164
197770124     163
177251122     162
203300524     154
             ... 
165780222       1
137010805       1
146180319       1
166930422       1
129960317       1
Name: count, Length: 3148, dtype: int64

In [20]:
class FeatureBuilder:
    def __init__(self, te_smooth=20, n_splits=5, seed=0):
        self.m, self.n_splits, self.seed = te_smooth, n_splits, seed

    # stage 1: row-wise features
    def _rows(self, df):
        inc = df["total_monthly_income"].clip(upper=self.inc_cap)
        yinc = df["yearly_total_income"].replace(0, np.nan)
        expn = df["yearly_total_expenses"].clip(upper=self.exp_cap)
        sav = df["yearly_total_savings"].clip(lower=-self.exp_cap, upper=self.exp_cap)
        fam = df["total_family_member"].clip(upper=12)
        fam_nz = fam.replace(0, np.nan)
        earn = df["total_earning_member"].clip(upper=8)
        earn_nz = earn.replace(0, np.nan)
        land = df["farm_land_amt"].where(df["farm_land_amt"] < 1e5)
        adm = pd.to_datetime(df["admission_date"])
        age = ((adm - pd.to_datetime(df["dateofbirth"])).dt.days / 365.25).clip(18, 70)
        pur = clean_text_purpose(df["loan_use_sources"])
        pur_c = pur.where(pur.isin(self.purposes), "rare")
        pg = pur.map(purpose_group)
        X = pd.DataFrame(index=df.index)
        #cleaned raw numeric
        X["monthly_income"] = inc
        X["land"] = land
        X["family_size"] = fam
        X["earners"] = earn
        X["yearly_expenses"] = expn
        X["yearly_savings"] = sav
        X["rent_years"] = df["year_of_rent"].clip(upper=60)
        X["age"] = age
        X["edu_ord"] = df["education"].map(EDU_MAP)
        X["adm_ord"] = (adm - pd.Timestamp("2017-01-01")).dt.days
        #engineered numeric
        X["exp_ratio"] = (df["yearly_total_expenses"] / yinc).clip(0, 3)
        X["sav_rate"] = (df["yearly_total_savings"] / yinc).clip(-3, 3)
        X["sav_mismatch"] = (df["yearly_total_savings"] != df["yearly_total_income"] - df["yearly_total_expenses"]).astype(int)
        X["inc_suspect"] = (X["exp_ratio"] < 0.10).astype(int)       # expenses < 10% of income: income likely overstated
        X["inc_per_member"] = inc / fam_nz
        X["inc_per_earner"] = inc / earn_nz
        X["earner_ratio"] = earn / fam_nz
        X["dependents"] = (fam - earn).clip(lower=0)
        X["dep_ratio"] = X["dependents"] / earn_nz
        X["land_zero"] = (df["farm_land_amt"] == 0).astype(int)
        X["land_per_member"] = land / fam_nz
        X["inc_round10k"] = (df["total_monthly_income"] % 10000 == 0).astype(int)
        X["inc_round50k"] = (df["total_monthly_income"] % 50000 == 0).astype(int)
        X["alt_flag"] = df["alternative_occupation"].notna().astype(int)
        X["home_missing"] = df["home_owner_type"].isna().astype(int)
        X["purpose_n"] = np.log1p(pur.map(self.purpose_counts).fillna(0))
        #categoricals
        def cat(s, cats): return pd.Categorical(s.where(s.isin(cats)), categories=cats)
        X["purpose"] = cat(pur_c, self.purposes + ["rare"])
        X["pgroup"] = cat(pg, self.pgroups)
        X["district"] = cat(df["current_addr_district"].astype(str).str.strip(), self.districts)
        X["thana"] = cat(df["current_addr_thana"].astype(str).str.strip(), self.thanas)
        X["occupation"] = cat(df["occupation"].fillna("NA").astype(str), self.occs)
        X["altocc"] = cat(df["alternative_occupation"].map(altocc_group), self.altoccs)
        X["home"] = cat(df["home_owner_type"].fillna("NA").astype(str), ["Own", "Rent", "NA"])
        X["marital"] = cat(df["maritial_status"].astype(str), self.maritals)
        X["female"] = (df["gender"] == "Female").astype(int)
        if "loan_Component" in df:
            c = df["loan_Component"].astype(str)
            c = np.where(c.str.startswith("AGROSOR"), "AGROSOR", np.where(c.isin(["JAGORON", "SUFOLON"]), c, "other"))
            X["comp"] = pd.Categorical(c, categories=["JAGORON", "SUFOLON", "AGROSOR", "other"])

        incband = pd.cut(inc, [-1, 30000, 40000, 50000, 60000, 70000, 80000, 100000, 1e12]).astype(str)
        K = pd.DataFrame({"prop": clean_id(df["proposer_oldid"]), "thana": df["current_addr_thana"].astype(str).str.strip(),
                          "district": df["current_addr_district"].astype(str).str.strip(), "purpose": pur_c.astype(str),
                          "altocc": X["altocc"].astype(str), "pur_inc": pur_c.astype(str) + "|" + incband,
                          "thana_pg": df["current_addr_thana"].astype(str).str.strip() + "|" + pg}, index=df.index)
        return X, K

    # peer statistics learned on training rows
    """
    Person Income high or low compared to same loan_amount, based on geographical area income high or low, common on property
    type, % on rented house, landownership differ from local avg
    """
    def _fit_peers(self, X, K):   #original numerical and categorical with grouping
        g = lambda key, col, fn: X[col].groupby(K[key].values).agg(fn)
        self.peer = {
            "pur_inc_med": g("purpose", "monthly_income", "median"),
            "thana_inc_med": g("thana", "monthly_income", "median"),
            "thana_land": g("thana", "land", "mean"),
            "thana_rent": X["home"].eq("Rent").groupby(K["thana"].values).mean(),
            "thana_n": K["thana"].value_counts(),
            "dist_inc_med": g("district", "monthly_income", "median"),
            "dist_home_missing": g("district", "home_missing", "mean"),
            "prop_n": K["prop"].value_counts(),
            "prop_inc_med": g("prop", "monthly_income", "median"),
        }
        self.glob = {"inc": float(X["monthly_income"].median()), "land": float(X["land"].mean())}

    def _apply_peers(self, X, K):
        P, G = self.peer, self.glob
        X["pur_inc_med"] = K["purpose"].map(P["pur_inc_med"]).fillna(G["inc"]).values
        X["thana_inc_med"] = K["thana"].map(P["thana_inc_med"]).fillna(G["inc"]).values
        X["thana_land"] = K["thana"].map(P["thana_land"]).fillna(G["land"]).values
        X["thana_rent"] = K["thana"].map(P["thana_rent"]).fillna(0).values
        X["thana_n"] = np.log1p(K["thana"].map(P["thana_n"]).fillna(0).values)
        X["dist_inc_med"] = K["district"].map(P["dist_inc_med"]).fillna(G["inc"]).values
        X["dist_home_missing"] = K["district"].map(P["dist_home_missing"]).fillna(0).values   # proxy for branch data-capture habit
        X["prop_n"] = np.log1p(K["prop"].map(P["prop_n"]).fillna(0).values)
        X["prop_inc_med"] = K["prop"].map(P["prop_inc_med"]).fillna(G["inc"]).values   # should used it or not, single proposer loanee can make issue
        X["inc_rel_purpose"] = (X["monthly_income"] / X["pur_inc_med"]).clip(0, 10)         # income vs median income with same purpose
        X["inc_rel_thana"] = (X["monthly_income"] / X["thana_inc_med"]).clip(0, 10)
        X["inc_rel_prop"] = (X["monthly_income"] / X["prop_inc_med"]).clip(0, 10)
        X["land_rel_thana"] = X["land"] / X["thana_land"].replace(0, np.nan)
        return X

    #target encodings
    TE_SPECS = {"te_prop": ("prop", "log"), "te_thana": ("thana", "log"), "te_district": ("district", "log"),
                "te_purpose": ("purpose", "log"), "te_altocc": ("altocc", "log"), "te_pur_inc": ("pur_inc", "log"),
                "te_thana_pg": ("thana_pg", "log"), "tb_prop": ("prop", "big"), "tb_thana": ("thana", "big"),
                "tb_pur_inc": ("pur_inc", "big")}

    def _te_table(self, key, target):     # smoothing the encoding
        return pd.DataFrame({"s": target, "k": key}).groupby("k")["s"].agg(["sum", "count"])

    def _te_apply(self, key, tab, prior):
        s = key.map(tab["sum"]).fillna(0).values   # sum of target value of  category
        c = key.map(tab["count"]).fillna(0).values  # number of observation on k category
        return (s + prior * self.m) / (c + self.m)   # p is global target mean and m is smoothing(how much trust the global avg)

    def _targets(self, y):
        return {"log": np.log1p(np.asarray(y, float)), "big": (np.asarray(y, float) >= BIG).astype(float)}

    #public API
    def fit_transform(self, df, y):
        d = df
        self.inc_cap = float(d["total_monthly_income"].quantile(0.999))
        self.exp_cap = float(d["yearly_total_expenses"].quantile(0.999))
        pur = clean_text_purpose(d["loan_use_sources"])
        vc = pur.value_counts()
        self.purposes = sorted(vc[vc >= 100].index.tolist())
        self.purpose_counts = vc
        self.pgroups = sorted(pur.map(purpose_group).unique().tolist())
        self.districts = sorted(d["current_addr_district"].astype(str).str.strip().unique().tolist())
        self.thanas = sorted(d["current_addr_thana"].astype(str).str.strip().unique().tolist())
        self.occs = sorted(d["occupation"].fillna("NA").astype(str).unique().tolist())
        self.altoccs = sorted(d["alternative_occupation"].map(altocc_group).unique().tolist())
        self.maritals = sorted(d["maritial_status"].astype(str).unique().tolist())
        X, K = self._rows(d)
        self._fit_peers(X, K)
        X = self._apply_peers(X, K)
        T = self._targets(y)
        self.prior = {k: float(v.mean()) for k, v in T.items()}
        self.te_tables = {name: self._te_table(K[key], T[kind]) for name, (key, kind) in self.TE_SPECS.items()}   #smoothing
        kf = KFold(self.n_splits, shuffle=True, random_state=self.seed)  # time based dataset  shuffle False, out-of-fold encodings for training rows
        for name, (key, kind) in self.TE_SPECS.items():
            out = np.zeros(len(X))
            for a, b in kf.split(X):
                tab = self._te_table(
                    K[key].iloc[a],
                    T[kind][a])

                out[b] = self._te_apply(
                    K[key].iloc[b],
                    tab,
                    self.prior[kind])
            X[name] = out
        return X


    def transform(self, df):
        X, K = self._rows(df)
        X = self._apply_peers(X, K)
        for name, (key, kind) in self.TE_SPECS.items():
            X[name] = self._te_apply(K[key], self.te_tables[name], self.prior[kind])
        return X

In [25]:
y = data["loan_amount"].values; n = len(data)
i1, i2 = int(n * 0.65), int(n * 0.80)
fb = FeatureBuilder()
Xa = fb.fit_transform(data.iloc[:i1], y[:i1]); Xv = fb.transform(data.iloc[i1:i2])
Xa

,monthly_income,land,family_size,earners,yearly_expenses,yearly_savings,rent_years,age,edu_ord,adm_ord,exp_ratio,sav_rate,sav_mismatch,inc_suspect,inc_per_member,inc_per_earner,earner_ratio,dependents,dep_ratio,land_zero,...,thana_rent,thana_n,dist_inc_med,dist_home_missing,prop_n,prop_inc_med,inc_rel_purpose,inc_rel_thana,inc_rel_prop,land_rel_thana,te_prop,te_thana,te_district,te_purpose,te_altocc,te_pur_inc,te_thana_pg,tb_prop,tb_thana,tb_pur_inc
0,100000.0,50.0,4,2,450000.0,750000.0,0.0,42.420260,4.0,3440,0.375000,0.625000,0,0,25000.000000,50000.0,0.500000,2,1.0,0,...,0.001873,6.974479,50000.0,0.231726,3.295837,100000.0,2.500000,2.000000,1.000000,1.590422,11.399998,11.250181,11.462612,11.160929,11.204140,12.019582,11.205195,0.084917,0.059589,0.309108
1,40000.0,90.0,4,1,320000.0,160000.0,0.0,29.782341,2.0,3423,0.666667,0.333333,0,0,10000.000000,40000.0,0.250000,3,3.0,0,...,0.008772,5.433722,60000.0,0.242726,3.258097,50000.0,0.800000,0.800000,0.800000,1.478812,11.397399,11.430495,11.410790,11.167548,11.204463,10.988935,11.314237,0.100982,0.097705,0.010095
2,450000.0,10.0,5,1,400000.0,5000000.0,0.0,37.505818,2.0,3420,0.074074,0.925926,0,1,90000.000000,450000.0,0.200000,4,4.0,0,...,0.000000,5.323010,45000.0,0.084146,3.912023,50000.0,10.000000,10.000000,9.000000,0.452128,11.140936,11.324628,11.280296,11.157462,11.203129,11.763358,11.264685,0.029429,0.115007,0.303555
3,80000.0,20.0,4,2,500000.0,460000.0,0.0,44.128679,3.0,3417,0.520833,0.479167,0,0,20000.000000,40000.0,0.500000,2,1.0,0,...,0.000000,5.758902,50000.0,0.165637,3.433987,60000.0,1.600000,1.600000,1.333333,0.449151,11.381989,11.398308,11.296703,11.375054,11.204463,11.735836,11.432309,0.156659,0.100136,0.276877
4,30000.0,200.0,3,1,144000.0,216000.0,0.0,43.789185,2.0,3287,0.400000,0.600000,0,0,10000.000000,30000.0,0.333333,2,2.0,0,...,0.000000,5.384495,35000.0,0.087361,3.761200,30000.0,0.750000,0.666667,1.000000,3.287879,10.997717,10.886260,10.909723,11.161926,11.202464,10.766586,10.857658,0.089365,0.024541,0.001761
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80590,30000.0,10.0,5,2,180000.0,180000.0,0.0,40.769336,2.0,2956,0.500000,0.500000,0,0,6000.000000,15000.0,0.400000,3,1.5,0,...,0.000000,5.739793,40000.0,0.200000,3.871201,30000.0,0.750000,0.845070,1.000000,0.300621,11.151365,11.052798,11.116786,10.518140,11.204798,10.554281,11.148704,0.092869,0.017739,0.022847
80591,90000.0,30.0,6,4,54000.0,1026000.0,0.0,40.175222,2.0,2930,0.050000,0.950000,0,1,15000.000000,22500.0,0.666667,2,0.5,0,...,0.028777,6.545350,55000.0,0.226619,3.135494,65000.0,2.000000,1.500000,1.384615,0.969542,11.414253,11.635356,11.493792,11.041903,11.204798,11.769724,11.239817,0.163895,0.178798,0.181045
80592,35000.0,100.0,4,1,300000.0,120000.0,0.0,37.941136,3.0,2900,0.714286,0.285714,0,0,8750.000000,35000.0,0.250000,3,3.0,0,...,0.009434,5.765191,40000.0,0.360934,3.332205,30000.0,0.875000,1.166667,1.166667,1.422819,11.195726,11.095698,11.051901,11.161926,11.184418,10.963073,11.073286,0.084917,0.051447,0.003994
80593,40000.0,10.0,3,2,30000.0,450000.0,0.0,47.030801,2.0,2933,0.062500,0.937500,0,1,13333.333333,20000.0,0.666667,1,0.5,0,...,0.000000,6.077642,50000.0,0.186944,3.496508,45000.0,0.888889,0.800000,0.888889,0.322724,11.065133,11.225787,11.239011,10.957210,11.179245,10.908313,11.631361,0.040380,0.064501,0.025814


In [30]:
GROUPS = {
    "base4": ["monthly_income", "purpose", "land", "family_size"],
    "raw_extra": ["earners", "yearly_expenses", "yearly_savings", "district", "thana", "edu_ord", "occupation",
                  "altocc", "home", "rent_years", "age"],
    "ratios": ["exp_ratio", "sav_rate", "sav_mismatch", "inc_suspect", "inc_per_member", "inc_per_earner", "earner_ratio",
               "dependents", "dep_ratio", "land_zero", "land_per_member", "inc_round10k", "inc_round50k", "alt_flag",
               "home_missing", "pgroup", "purpose_n"],
    "peers": ["pur_inc_med", "thana_inc_med", "thana_land", "thana_rent", "thana_n", "dist_inc_med", "dist_home_missing",
              "prop_n", "prop_inc_med", "inc_rel_purpose", "inc_rel_thana", "inc_rel_prop", "land_rel_thana"],
    "target_enc": list(FeatureBuilder.TE_SPECS),
    "time": ["adm_ord"],
    "sensitive": ["female", "marital"],
    "component": ["comp"],
}

In [43]:
def candidate_pool(include_component=False, include_sensitive=False, include_time=True):
    pool = sum([GROUPS[g] for g in ["base4", "raw_extra", "ratios", "peers", "target_enc"]], [])
    if include_time: pool += GROUPS["time"]
    if include_sensitive: pool += GROUPS["sensitive"]
    if include_component: pool += GROUPS["component"]
    return list(dict.fromkeys(pool))

In [45]:
candidate_pool()

['monthly_income',
 'purpose',
 'land',
 'family_size',
 'earners',
 'yearly_expenses',
 'yearly_savings',
 'district',
 'thana',
 'edu_ord',
 'occupation',
 'altocc',
 'home',
 'rent_years',
 'age',
 'exp_ratio',
 'sav_rate',
 'sav_mismatch',
 'inc_suspect',
 'inc_per_member',
 'inc_per_earner',
 'earner_ratio',
 'dependents',
 'dep_ratio',
 'land_zero',
 'land_per_member',
 'inc_round10k',
 'inc_round50k',
 'alt_flag',
 'home_missing',
 'pgroup',
 'purpose_n',
 'pur_inc_med',
 'thana_inc_med',
 'thana_land',
 'thana_rent',
 'thana_n',
 'dist_inc_med',
 'dist_home_missing',
 'prop_n',
 'prop_inc_med',
 'inc_rel_purpose',
 'inc_rel_thana',
 'inc_rel_prop',
 'land_rel_thana',
 'te_prop',
 'te_thana',
 'te_district',
 'te_purpose',
 'te_altocc',
 'te_pur_inc',
 'te_thana_pg',
 'tb_prop',
 'tb_thana',
 'tb_pur_inc',
 'adm_ord']

#### Model Initialized

In [46]:
PARAMS = dict(learning_rate=0.08, num_leaves=31, min_child_samples=50, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.8, reg_lambda=5, cat_smooth=30, min_data_per_group=50, cat_l2=10,
              max_cat_to_onehot=4, n_estimators=2500, verbose=-1, n_jobs=1)

def fit_model(X, y_log, feats, objective="regression", seed=0, **kw):
    """Early-stops on the most recent 15% of the rows given (time-ordered)."""
    p = dict(PARAMS, objective=objective, random_state=seed)
    p.update(kw)
    if objective == "huber": p.setdefault("alpha", 0.7)
    k = int(len(X) * 0.85)
    m = lgb.LGBMRegressor(**p)
    m.fit(X.iloc[:k][feats], y_log[:k], eval_set=[(X.iloc[k:][feats], y_log[k:])],
          eval_metric="l2" if objective == "regression" else "l1",
          callbacks=[lgb.early_stopping(50, verbose=False)])
    return m

def predict_taka(m, X, feats):
    return np.expm1(m.predict(X[feats]))

### Feature Selection

In [53]:
df = load()
y = df["loan_amount"].values
n = len(df)
i1, i2 = int(n * 0.65), int(n * 0.80)
fb = FeatureBuilder()
Xa = fb.fit_transform(df.iloc[:i1], y[:i1])
Xv = fb.transform(df.iloc[i1:i2])
ya, yv = np.log1p(y[:i1]), y[i1:i2]
print(f"fit rows {i1:,} | validation rows {i2 - i1:,}", flush=True)

fit rows 80,595 | validation rows 18,599


In [55]:
def val_mae(m, feats, X=Xv, yy=yv):
    return float(np.mean(np.abs(np.clip(np.expm1(m.predict(X[feats])), 4000, 5e6) - yy)))

In [30]:
for track in ["A", "B"]:
    t0 = time.time()
    pool = candidate_pool(include_component=(track == "B"), include_sensitive=False)   # B with component
    #filter
    num = [c for c in pool if str(Xa[c].dtype) != "category"]
    sample = Xa[num].sample(20000, random_state=0)       #exclude categorical features, monotonic
    corr = sample.corr(method="spearman").abs()
    drop = set()
    for i, a in enumerate(num):
        for b in num[i + 1:]:
            if a not in drop and b not in drop and corr.loc[a, b] > 0.97:
                drop.add(b)
                print(f"[{track}] near-duplicate: dropping {b} (|rho|={corr.loc[a, b]:.3f} with {a})")

    const = [c for c in pool if Xa[c].nunique(dropna=False) <= 1]    #unchange feature
    feats = [c for c in pool if c not in drop and c not in const]
    print(f"[{track}] pool {len(pool)} -> {len(feats)} after filtering", flush=True)

    #full model + permutation importance
    full = fit_model(Xa, ya, feats)   #train model
    base_mae = val_mae(full, feats)
    print(f"[{track}] ALL {len(feats)} features: val MAE {base_mae:,.0f} (iters {full.best_iteration_}, {time.time() - t0:.0f}s)", flush=True)

    rng = np.random.RandomState(0)
    sub = Xv.iloc[:12000].copy()
    ysub = yv[:12000]
    base_sub = val_mae(full, feats, sub, ysub)
    imp = {}
    for f in feats:
        inc = []
        for r in range(2):
            Xp = sub.copy()
            Xp[f] = Xp[f].values[rng.permutation(len(Xp))]
            inc.append(val_mae(full, feats, Xp, ysub) - base_sub)    #The larger the positive value, the more the model depends on that feature
        imp[f] = float(np.mean(inc))    #validation
    gain = pd.Series(full.booster_.feature_importance("gain"), index=feats)
    gain = gain / gain.sum()                                                #training
    rank = pd.DataFrame({"perm_importance_MAE": pd.Series(imp), "gain_share": gain}).sort_values("perm_importance_MAE", ascending=False)
    rank.round(4).to_csv(os.path.join(OUT, f"importance_track{track}.csv"))
    print(rank.round(4).head(20).to_string(), flush=True)
    # choose k
    order = rank.index.tolist()
    results = []
    for k in [6, 10, 14, 18, 24, 32, len(feats)]:
        k = min(k, len(feats)); sel = order[:k]
        m = fit_model(Xa, ya, sel); mm = metrics(yv, np.expm1(m.predict(Xv[sel])))
        results.append({"k": k, "val_MAE": mm["MAE"], "val_R2_log": mm["R2_log"], "val_within1": mm["within_1_step"]})
        print(f"[{track}] top-{k:>2}: val MAE {mm['MAE']:,.0f}  R2_log {mm['R2_log']:.3f}  ({time.time() - t0:.0f}s)", flush=True)
    res = pd.DataFrame(results).drop_duplicates("k")
    best = res.val_MAE.min()
    k_star = int(res[res.val_MAE <= best * 1.003].k.min())
    selected = order[:k_star]
    print(f"[{track}] chosen k = {k_star}: {selected}", flush=True)
    json.dump({"track": track, "selected": selected, "k_curve": res.round(4).to_dict("records"), "dropped_duplicates": sorted(drop)}, open(os.path.join(OUT, f"selection_track{track}.json"), "w"), indent=2)

[A] near-duplicate: dropping land_per_member (|rho|=0.993 with land)
[A] near-duplicate: dropping dep_ratio (|rho|=1.000 with earner_ratio)
[A] pool 56 -> 54 after filtering
[A] ALL 54 features: val MAE 38,998 (iters 261, 4s)
                 perm_importance_MAE  gain_share
te_pur_inc                 7231.8629      0.3475
inc_rel_prop               5252.7484      0.0561
tb_pur_inc                 3070.1385      0.2207
te_prop                    1871.7725      0.0363
thana                      1481.2490      0.0816
monthly_income              974.4922      0.0419
purpose                     753.8671      0.0240
district                    573.3172      0.0223
edu_ord                     555.2859      0.0046
land_rel_thana              459.8332      0.0143
yearly_savings              451.1396      0.0057
exp_ratio                   437.9988      0.0080
te_thana_pg                 436.8597      0.0263
tb_prop                     363.1923      0.0065
yearly_expenses             329.4029   

### FE Final

In [65]:
SE={t: json.load(open(os.path.join(OUT, f"selection_track{t}.json"))) for t in "AB"}
print(SE)

{'A': {'track': 'A', 'selected': ['te_pur_inc', 'inc_rel_prop', 'tb_pur_inc', 'te_prop', 'thana', 'monthly_income', 'purpose', 'district', 'edu_ord', 'land_rel_thana', 'yearly_savings', 'exp_ratio', 'te_thana_pg', 'tb_prop', 'yearly_expenses', 'inc_rel_purpose', 'inc_rel_thana', 'te_purpose', 'occupation', 'land', 'inc_suspect', 'thana_rent', 'prop_n', 'prop_inc_med', 'inc_per_earner', 'te_altocc', 'inc_round10k', 'inc_per_member', 'sav_rate', 'te_district', 'tb_thana', 'sav_mismatch'], 'k_curve': [{'k': 6, 'val_MAE': 41212.0686, 'val_R2_log': 0.5819, 'val_within1': 0.4858}, {'k': 10, 'val_MAE': 40108.7339, 'val_R2_log': 0.6023, 'val_within1': 0.494}, {'k': 14, 'val_MAE': 39447.5453, 'val_R2_log': 0.6118, 'val_within1': 0.4977}, {'k': 18, 'val_MAE': 39245.3672, 'val_R2_log': 0.6153, 'val_within1': 0.501}, {'k': 24, 'val_MAE': 39148.3518, 'val_R2_log': 0.6168, 'val_within1': 0.4988}, {'k': 32, 'val_MAE': 38911.4518, 'val_R2_log': 0.6175, 'val_within1': 0.4998}, {'k': 54, 'val_MAE': 3891

In [64]:
print(SE['A']['selected'])
print(SE['B']['dropped_duplicates'])

['te_pur_inc', 'inc_rel_prop', 'tb_pur_inc', 'te_prop', 'thana', 'monthly_income', 'purpose', 'district', 'edu_ord', 'land_rel_thana', 'yearly_savings', 'exp_ratio', 'te_thana_pg', 'tb_prop', 'yearly_expenses', 'inc_rel_purpose', 'inc_rel_thana', 'te_purpose', 'occupation', 'land', 'inc_suspect', 'thana_rent', 'prop_n', 'prop_inc_med', 'inc_per_earner', 'te_altocc', 'inc_round10k', 'inc_per_member', 'sav_rate', 'te_district', 'tb_thana', 'sav_mismatch']
['dep_ratio', 'land_per_member']


In [69]:
df = load()
y = df["loan_amount"].values
n = len(df)
cut = int(n * 0.8)
sel = {t: json.load(open(os.path.join(OUT, f"selection_track{t}.json"))) for t in "AB"}
SEL_A, SEL_B = sel["A"]["selected"], sel["B"]["selected"]
dups = set(sel["A"]["dropped_duplicates"])
t0 = time.time()
fb = FeatureBuilder()
Xtr = fb.fit_transform(df.iloc[:cut], y[:cut])
Xte = fb.transform(df.iloc[cut:])
ytr, yte = np.log1p(y[:cut]), y[cut:]
print(f"train {cut:,} | test {n - cut:,} | features built ({time.time() - t0:.0f}s)", flush=True)

train 99,194 | test 24,799 | features built (2s)


In [70]:
def run(feats, objective="regression", seeds=(0, 1, 2), **kw):
    ms, preds = [], []
    for s in seeds:
        m = fit_model(Xtr, ytr, feats, objective, seed=s, **kw)
        p = predict_taka(m, Xte, feats)
        preds.append(p)
        ms.append(metrics(yte, p))
    r = pd.DataFrame(ms)
    out = r.mean().to_dict()
    out["MAE_sd"], out["R2_log_sd"], out["iters"] = r.MAE.std(), r.R2_log.std(), m.best_iteration_
    out["n_features"] = len(feats)
    return out, np.mean(preds, axis=0), m

In [73]:
# staged comparison
G = GROUPS
clean = lambda L: [f for f in dict.fromkeys(L) if f not in dups]
stages = {
    "S0  base 4 features, L1 (original base model)": (G["base4"], "l1"),
    "S0b base 4 features, L2": (G["base4"], "regression"),
    "S1  + cleaned raw columns": (clean(G["base4"] + G["raw_extra"]), "regression"),
    "S2  + ratios / flags / purpose group": (clean(G["base4"] + G["raw_extra"] + G["ratios"]), "regression"),
    "S3  + peer statistics": (clean(G["base4"] + G["raw_extra"] + G["ratios"] + G["peers"]), "regression"),
    "S4  + target encodings (all 54)": (clean(G["base4"] + G["raw_extra"] + G["ratios"] + G["peers"] + G["target_enc"] + G["time"]), "regression"),
    "S5  SELECTED, track A (no component)": (SEL_A, "regression"),
    "S6  SELECTED, track B (+ loan_Component)": (SEL_B, "regression"),
}

rows = {}
for name, (feats, obj) in stages.items():
    r, _, _ = run(feats, obj)
    rows[name] = r
    print(f"{name:48s} n_feat {r['n_features']:>2}  MAE {r['MAE']:>8,.0f} (±{r['MAE_sd']:.0f})  R2_log {r['R2_log']:.3f} (±{r['R2_log_sd']:.3f})  R2_taka {r['R2_taka']:.3f}  within1 {r['within_1_step']:.3f}", flush=True)
T = pd.DataFrame(rows).T
T.round(4).to_csv(os.path.join(OUT, "stages.csv"))

S0  base 4 features, L1 (original base model)    n_feat  4  MAE   41,112 (±38)  R2_log 0.529 (±0.001)  R2_taka 0.490  within1 0.471
S0b base 4 features, L2                          n_feat  4  MAE   41,298 (±15)  R2_log 0.540 (±0.000)  R2_taka 0.497  within1 0.464
S1  + cleaned raw columns                        n_feat 15  MAE   37,670 (±53)  R2_log 0.607 (±0.001)  R2_taka 0.610  within1 0.501
S2  + ratios / flags / purpose group             n_feat 30  MAE   37,553 (±45)  R2_log 0.608 (±0.000)  R2_taka 0.616  within1 0.502
S3  + peer statistics                            n_feat 43  MAE   36,912 (±29)  R2_log 0.621 (±0.001)  R2_taka 0.627  within1 0.514
S4  + target encodings (all 54)                  n_feat 54  MAE   36,529 (±66)  R2_log 0.626 (±0.001)  R2_taka 0.637  within1 0.517
S5  SELECTED, track A (no component)             n_feat 32  MAE   36,636 (±60)  R2_log 0.624 (±0.000)  R2_taka 0.631  within1 0.516
S6  SELECTED, track B (+ loan_Component)         n_feat 18  MAE   30,494 (±6

In [35]:
# ablation by role
def role(f):
    if "prop" in f: return "officer (proposer)"
    if f in ("thana", "district", "te_thana", "tb_thana", "te_district", "te_thana_pg") or f.startswith(("thana_", "dist_")) or f in ("land_rel_thana", "inc_rel_thana"): return "geography"
    if "purpose" in f or f in ("pgroup", "te_pur_inc", "tb_pur_inc", "te_altocc", "altocc", "purpose_n"): return "purpose"
    if f in ("land", "edu_ord", "occupation", "family_size", "earners", "age", "rent_years", "home", "dependents", "earner_ratio", "land_zero", "alt_flag", "home_missing"): return "household / profile"
    if f == "comp": return "component"
    if f == "adm_ord": return "time"
    return "money (income, expenses, savings)"
base_r, _, _ = run(SEL_A, seeds=(0, 1))
ab = []
for rl in sorted({role(f) for f in SEL_A}):
    keep = [f for f in SEL_A if role(f) != rl]
    r, _, _ = run(keep, seeds=(0, 1))
    ab.append([rl, sum(role(f) == rl for f in SEL_A), r["MAE"] - base_r["MAE"], r["R2_log"] - base_r["R2_log"]])
    print(f"drop {rl:36s} MAE {ab[-1][2]:+8,.0f}  R2_log {ab[-1][3]:+.4f}", flush=True)
pd.DataFrame(ab, columns=["role_dropped", "n_features_dropped", "dMAE", "dR2_log"]).round(4).to_csv(os.path.join(OUT, "ablation_roles.csv"), index=False)

drop geography                            MAE     +623  R2_log -0.0119
drop household / profile                  MAE     +191  R2_log -0.0037
drop money (income, expenses, savings)    MAE     +593  R2_log -0.0091
drop officer (proposer)                   MAE   +1,035  R2_log -0.0187
drop purpose                              MAE     +999  R2_log -0.0210


In [74]:
# # objectives (default params), penalize and linear
for obj in ["regression", "huber", "l1"]:
    r, _, _ = run(SEL_A, obj); print(f"objective {obj:10s} MAE {r['MAE']:,.0f}  R2_log {r['R2_log']:.3f}  RMSLE {r['RMSLE']:.3f}  within1 {r['within_1_step']:.3f}", flush=True)

In [37]:
# tuning on VALIDATION only
i1 = int(n * 0.65)
fb2 = FeatureBuilder()
Xa = fb2.fit_transform(df.iloc[:i1], y[:i1])
Xv = fb2.transform(df.iloc[i1:cut])
ya, yv = np.log1p(y[:i1]), y[i1:cut]
space = dict(num_leaves=[15, 31, 63], min_child_samples=[20, 50, 100, 200], colsample_bytree=[0.5, 0.7, 0.9],
             reg_lambda=[1, 5, 20, 50], cat_smooth=[10, 30, 60], min_data_per_group=[20, 50, 100], cat_l2=[10, 30])
rs = np.random.RandomState(1)
trials = [{}] + [{k: rs.choice(v).item() for k, v in space.items()} for _ in range(14)]
best = {}
for track, feats in [("A", SEL_A), ("B", SEL_B)]:
    res = []
    for i, kw in enumerate(trials):
        m = fit_model(Xa, ya, feats, "regression", **kw)
        mm = metrics(yv, np.expm1(m.predict(Xv[feats])))
        res.append((mm["R2_log"], mm["MAE"], kw)); print(f"[tune {track}] trial {i:>2} val R2_log {mm['R2_log']:.4f} MAE {mm['MAE']:,.0f} {kw}", flush=True)
    best[track] = max(res, key=lambda r: r[0])[2]; print(f"[tune {track}] BEST {best[track]}", flush=True)

[tune A] trial  0 val R2_log 0.6175 MAE 38,911 {}
[tune A] trial  1 val R2_log 0.6211 MAE 38,922 {'num_leaves': 31, 'min_child_samples': 200, 'colsample_bytree': 0.5, 'reg_lambda': 1, 'cat_smooth': 30, 'min_data_per_group': 50, 'cat_l2': 30}
[tune A] trial  2 val R2_log 0.6173 MAE 39,072 {'num_leaves': 15, 'min_child_samples': 20, 'colsample_bytree': 0.7, 'reg_lambda': 1, 'cat_smooth': 30, 'min_data_per_group': 20, 'cat_l2': 10}
[tune A] trial  3 val R2_log 0.6212 MAE 38,810 {'num_leaves': 31, 'min_child_samples': 100, 'colsample_bytree': 0.5, 'reg_lambda': 20, 'cat_smooth': 30, 'min_data_per_group': 100, 'cat_l2': 10}
[tune A] trial  4 val R2_log 0.6226 MAE 38,843 {'num_leaves': 15, 'min_child_samples': 100, 'colsample_bytree': 0.5, 'reg_lambda': 5, 'cat_smooth': 60, 'min_data_per_group': 100, 'cat_l2': 10}
[tune A] trial  5 val R2_log 0.6191 MAE 38,907 {'num_leaves': 31, 'min_child_samples': 50, 'colsample_bytree': 0.9, 'reg_lambda': 1, 'cat_smooth': 60, 'min_data_per_group': 50, 'ca

In [38]:
# final honest test scores
final = {}; preds = {}
for track, feats in [("A", SEL_A), ("B", SEL_B)]:
    for obj, tag in [("regression", "L2"), ("l1", "L1")]:
        r, p, m = run(feats, obj, **best[track]); final[f"track {track} tuned {tag}"] = r
        preds[(track, tag)] = p
        print(f"FINAL track {track} {tag}: MAE {r['MAE']:,.0f} (±{r['MAE_sd']:.0f})  R2_log {r['R2_log']:.3f}  R2_taka {r['R2_taka']:.3f}  RMSLE {r['RMSLE']:.3f}  within1 {r['within_1_step']:.3f}  exact {r['exact_step']:.3f}  MAE_big {r['MAE_big(>=300k)']:,.0f}  AUC_big {r['AUC_big']:.3f}", flush=True)
pd.DataFrame(final).T.round(4).to_csv(os.path.join(OUT, "final_metrics.csv"))
json.dump({"A": best["A"], "B": best["B"], "selected_A": SEL_A, "selected_B": SEL_B}, open(os.path.join(OUT, "best_params.json"), "w"), indent=2)
# gain importance of the final track-A L2 model
r, p, m = run(SEL_A, "regression", seeds=(0,), **best["A"])
g = pd.Series(m.booster_.feature_importance("gain"), index=SEL_A)
(g / g.sum()).sort_values(ascending=False).round(4).to_csv(os.path.join(OUT, "final_gain_trackA.csv"))
# test predictions
out = pd.DataFrame({"member_id": df["member_id"].iloc[cut:].values, "actual_loan": yte,
                    "pred_A_L2": preds[("A", "L2")].round(0), "pred_A_L1": preds[("A", "L1")].round(0),
                    "pred_B_L2": preds[("B", "L2")].round(0), "pred_B_L1": preds[("B", "L1")].round(0),
                    "gender": df["gender"].iloc[cut:].values, "district": df["current_addr_district"].iloc[cut:].values})
out.to_csv(os.path.join(OUT, "test_predictions_fe.csv"), index=False)

FINAL track A L2: MAE 36,285 (±59)  R2_log 0.628  R2_taka 0.638  RMSLE 0.446  within1 0.518  exact 0.201  MAE_big 148,238  AUC_big 0.960
FINAL track A L1: MAE 36,244 (±101)  R2_log 0.624  R2_taka 0.636  RMSLE 0.449  within1 0.527  exact 0.219  MAE_big 143,419  AUC_big 0.958
FINAL track B L2: MAE 30,258 (±51)  R2_log 0.694  R2_taka 0.760  RMSLE 0.405  within1 0.555  exact 0.220  MAE_big 94,174  AUC_big 0.999
FINAL track B L1: MAE 29,935 (±101)  R2_log 0.689  R2_taka 0.754  RMSLE 0.408  within1 0.559  exact 0.246  MAE_big 91,098  AUC_big 0.999


In [43]:
#refit on ALL rows and save deployable models
# fb_all = FeatureBuilder(); Xall = fb_all.fit_transform(df, y); yall = np.log1p(y)
# meta = {"features_A": SEL_A, "features_B": SEL_B}
# for track, feats in [("A", SEL_A), ("B", SEL_B)]:
#     iters = int(final[f"track {track} tuned L2"]["iters"] * 1.1) + 1
#     p = dict(PARAMS, objective="regression", random_state=0, n_estimators=iters); p.update(best[track])
#     mdl = lgb.LGBMRegressor(**p).fit(Xall[feats], yall)
#     mdl.booster_.save_model(os.path.join(MODELS, f"model_{track}.txt")); meta[f"n_estimators_{track}"] = iters
# pickle.dump(fb_all, open(os.path.join(MODELS, "feature_builder.pkl"), "wb"))
# json.dump(meta, open(os.path.join(MODELS, "meta.json"), "w"), indent=2)
# print(f"saved models -> {MODELS}   total time {time.time() - t0:.0f}s", flush=True)

saved models -> ./models   total time 2121s


### Predict

In [45]:
# import lightgbm as lgb
# import numpy as np
# import pandas as pd
#
# HERE = os.getcwd()
# sys.path.insert(0, HERE)
# LADDER = np.array([5, 10, 20, 25, 30, 40, 50, 60, 70, 80, 90, 100, 120, 150, 200, 250, 300, 400, 500, 800, 1000, 2000, 5000]) * 1000
# REQUIRED = ["admission_date", "dateofbirth", "gender", "maritial_status", "education", "occupation", "alternative_occupation",
#             "home_owner_type", "year_of_rent", "current_addr_district", "current_addr_thana", "proposer_oldid",
#             "loan_use_sources", "farm_land_amt", "total_family_member", "total_earning_member", "total_monthly_income",
#             "yearly_total_income", "yearly_total_expenses", "yearly_total_savings"]

In [1]:
# def predict(df_new: pd.DataFrame, track: str = "A", model_dir: str = os.path.join(HERE, "models")) -> pd.DataFrame:
#     need = REQUIRED + (["loan_Component"] if track == "B" else [])
#     missing = [c for c in need if c not in df_new.columns]
#     if missing:
#         raise ValueError(f"missing required columns: {missing}")
#     fb = pickle.load(open(os.path.join(model_dir, "feature_builder.pkl"), "rb"))
#     meta = json.load(open(os.path.join(model_dir, "meta.json")))
#     booster = lgb.Booster(model_file=os.path.join(model_dir, f"model_{track}.txt"))
#     X = fb.transform(df_new)
#     pred = np.clip(np.expm1(booster.predict(X[meta[f"features_{track}"]])), 4000, 5e6)
#     near = LADDER[np.abs(np.log(pred)[:, None] - np.log(LADDER)[None, :]).argmin(1)]
#     out = pd.DataFrame({"predicted_loan": pred.round(0), "nearest_standard_amount": near}, index=df_new.index)
#     if "member_id" in df_new:
#         out.insert(0, "member_id", df_new["member_id"].values)
#     return out
#
# if __name__ == "__main__":
#     csv, track, dest = sys.argv[1], sys.argv[2], sys.argv[3]
#     res = predict(pd.read_csv(csv), track)
#     res.to_csv(dest, index=False); print(f"wrote {len(res):,} predictions -> {dest}")

In [ ]:
# csv, track, dest = sys.argv[1], sys.argv[2], sys.argv[3]